In [ ]:
# Cell 1 — Imports
# Run this first every time you open the notebook

import pandas as pd
import numpy as np
from pathlib import Path

# pandas   = read and manipulate CSV files
# numpy    = maths on arrays
# pathlib  = file path handling

print("Imports OK")

In [ ]:
# Cell 2 — Load the main CSV file
# Run once — takes 30-60 seconds for 2.8M rows

CSV_PATH = (
    r"J:\- Macros\AI-LaneDetector\lane_analysis"
    r"\coordinates_report_with_lane_combined_with_ignore_full1.csv"
)

df = pd.read_csv(CSV_PATH, low_memory=False)
# low_memory=False = read whole file before deciding column types
# fixes the mixed type warning

print(f"Total rows loaded: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLane distribution:")
print(df["Lane"].value_counts().to_string())

In [ ]:
# Cell 3 — Clean the data
# Remove ignore rows and unknown lane classes

# Step 1: define which classes we keep (Phase 1)
LANE_CLASSES = {"1", "2", "3", "SK1"}
# using a set for fast lookup
# Java equivalent: Set<String> laneClasses = new HashSet<>()

# Step 2: remove ignored frames
df_clean = df[df["Ignore"] != 1.0].copy()
print(f"After removing ignore rows: {len(df_clean):,}")

# Step 3: keep only known lane classes
df_clean = df_clean[df_clean["Lane"].isin(LANE_CLASSES)].copy()
print(f"After keeping only known classes: {len(df_clean):,}")

# Step 4: extract segment from filepath
df_clean["SourceFolder"] = df_clean["Filepath"].str.extract(
    r"Testing\\([^\\]+)\\"
)
# extracts the project folder name from the filepath
# e.g. "J:\Testing\250816 SouthlandDC...\photo.jpg"
# → "250816 SouthlandDC_NetworkLMD_25"

print(f"\nSegments found:")
print(df_clean["SourceFolder"].value_counts().to_string())

print(f"\nFinal lane distribution:")
for lane in sorted(LANE_CLASSES):
    count = (df_clean["Lane"] == lane).sum()
    pct   = count / len(df_clean) * 100
    print(f"  {lane:>4}: {count:>10,}  ({pct:.2f}%)")

In [ ]:
# Cell 4 — Split into train / val / test by road segment
# Each segment is a complete survey run
# We never mix frames from the same segment across splits
# This prevents the model from "memorising" sequences

TEST_SEGMENTS = [
    "250357 UHCC_25_LMD",
]
# UHCC has 87,014 rows — good size for test set

VAL_SEGMENTS = [
    "250821 KaikouraDC_Network25_LMD Demo",
    "250846 DunedinCC LMD Network26",
]
# DunedinCC has Lane 3 (685 rows) ← important
# KaikouraDC adds more val data
# DunedinCC included because it has Lane 3

# everything not in test or val goes to train
train_df = df_clean[
    ~df_clean["SourceFolder"].isin(TEST_SEGMENTS + VAL_SEGMENTS)
].copy()
# ~ = NOT operator
# .isin() checks if value is in the list
# together: keep rows where segment is NOT in test or val

val_df = df_clean[
    df_clean["SourceFolder"].isin(VAL_SEGMENTS)
].copy()

test_df = df_clean[
    df_clean["SourceFolder"].isin(TEST_SEGMENTS)
].copy()

# print summary
print(f"Train: {len(train_df):,} rows")
print(f"Val:   {len(val_df):,} rows")
print(f"Test:  {len(test_df):,} rows")
print(f"Total: {len(train_df) + len(val_df) + len(test_df):,} rows")

# check all 4 classes appear in each split
for name, split in [("Train", train_df),
                    ("Val",   val_df),
                    ("Test",  test_df)]:
    print(f"\n{name} lane distribution:")
    for lane in ["1", "2", "3", "SK1"]:
        count = (split["Lane"] == lane).sum()
        pct   = count / len(split) * 100 if len(split) > 0 else 0
        flag  = " ← MISSING" if count == 0 else ""
        print(f"  {lane:>4}: {count:>10,}  ({pct:.2f}%){flag}")

In [ ]:
# Cell 5 — Save train/val/test CSV files
# Only run this once when you are happy with the splits
# These files go in the Data/ folder inside geosolve_lanes

OUTPUT_DIR = Path(
    r"C:\Users\gd\New folder\project"
    r"\geosolve_lanes\geosolve_lanes\Data"
)
# adjust this path if your project is in a different location

OUTPUT_DIR.mkdir(exist_ok=True)
# create Data/ folder if it doesn't exist

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(  OUTPUT_DIR / "val.csv",   index=False)
test_df.to_csv( OUTPUT_DIR / "test.csv",  index=False)
# index=False = don't write row numbers as a column

print(f"Saved to {OUTPUT_DIR}")
print(f"  train.csv: {len(train_df):,} rows")
print(f"  val.csv:   {len(val_df):,} rows")
print(f"  test.csv:  {len(test_df):,} rows")

In [ ]:
# Cell 6 — Verify saved files loaded correctly
# Run this to double-check everything saved properly

train_check = pd.read_csv(OUTPUT_DIR / "train.csv")
val_check   = pd.read_csv(OUTPUT_DIR / "val.csv")
test_check  = pd.read_csv(OUTPUT_DIR / "test.csv")

print("Files verified:")
print(f"  train.csv: {len(train_check):,} rows, "
      f"columns: {train_check.columns.tolist()}")
print(f"  val.csv:   {len(val_check):,} rows")
print(f"  test.csv:  {len(test_check):,} rows")
print("\nAll good — ready to train!")

In [ ]:
# Cell 7 — Verify images are accessible from J:\ drive
# Run before starting training to confirm files load correctly

import cv2

sample_paths = train_check["Filepath"].head(10).tolist()
# grab 10 filepaths from training set

success = 0
failed  = 0

for filepath in sample_paths:
    filepath = str(filepath).replace("\\", "/")
    img = cv2.imread(filepath)
    if img is not None:
        success += 1
    else:
        failed += 1
        print(f"FAILED: {filepath}")

print(f"\nImage access check:")
print(f"  Loaded:  {success}/10")
print(f"  Failed:  {failed}/10")

if success == 10:
    print("  ✓ J:\\ drive accessible, images load correctly")
    img_example = cv2.imread(
        str(sample_paths[0]).replace("\\", "/")
    )
    print(f"  Image shape: {img_example.shape}")
    # should be something like (2200, 3208, 3)
    # height x width x 3 colour channels
else:
    print("  ✗ Some images failed — check J:\\ drive connection")

In [ ]:
# Cell 8 — Estimate how long training will take
# Run before starting overnight training

import torch
import time
import sys
sys.path.append(
    r"C:\Users\gd\New folder\project\geosolve_lanes\geosolve_lanes"
)
# add project folder to path so we can import our files

from dataset import get_train_loader
from model   import get_model

train_loader = get_train_loader(batch_size=32, num_workers=0)
model        = get_model(pretrained=False)
# pretrained=False = no download needed just for timing
model.eval()

print("Timing 10 batches...")
times = []

for batch_idx, (images, gps, labels) in enumerate(train_loader):
    start = time.time()
    with torch.no_grad():
        output = model(images, gps)
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"  Batch {batch_idx+1}: {elapsed:.2f}s")
    if batch_idx >= 9:
        break

avg  = sum(times) / len(times)
# average seconds per batch

batches_per_epoch = len(train_loader)
secs_per_epoch    = avg * batches_per_epoch
hours_per_epoch   = secs_per_epoch / 3600

print(f"\nResults:")
print(f"  Avg time per batch:     {avg:.2f} seconds")
print(f"  Batches per epoch:      {batches_per_epoch:,}")
print(f"  Estimated time/epoch:   {hours_per_epoch:.1f} hours")
print(f"  Phase 1 (5 epochs):     {hours_per_epoch*5:.1f} hours")
print(f"  Phase 2 (30 epochs):    {hours_per_epoch*30:.1f} hours")
print(f"  Total estimate:         {hours_per_epoch*35:.1f} hours")

if hours_per_epoch > 8:
    print("\n  ⚠ WARNING: Training will take very long on CPU")
    print("  Consider asking your boss about GPU access")
    print("  Or reduce batch to process fewer images per epoch")

In [ ]:
print(df_clean["SourceFolder"].value_counts().to_string())

In [ ]:
# In data_preparation.ipynb add a new cell:
# Take only 10% of training data randomly
train_small = train_df.sample(frac=0.1, random_state=42)
train_small.to_csv(OUTPUT_DIR / "train_small.csv", index=False)
print(f"Small train set: {len(train_small):,} rows")
# → ~229,000 rows instead of 2.3M
# → 7,175 batches per epoch instead of 71,616
# → ~1.8 hours per epoch instead of 18
# → Phase 1 = 9 hours ← overnight feasible

In [1]:
import pandas as pd

df = pd.read_csv("Data/train_small.csv")

# stratified sample — 125 per class max
samples = []
for lane in df["Lane"].unique():
    subset = df[df["Lane"] == lane]
    samples.append(subset.sample(min(len(subset), 125), random_state=42))

tiny = pd.concat(samples).reset_index(drop=True)
print(f"Tiny train: {len(tiny):,} rows")
print(tiny["Lane"].value_counts().to_string())
tiny.to_csv("Data/train_tiny.csv", index=False)
print("saved!")

Tiny train: 500 rows
Lane
1      125
2      125
3      125
SK1    125
saved!


In [2]:
import pandas as pd

df = pd.read_csv("Data/val.csv")

# stratified sample of 1000 rows — enough for local testing
samples = []
for lane in df["Lane"].unique():
    subset = df[df["Lane"] == lane]
    samples.append(subset.sample(min(len(subset), 250), random_state=42))

val_small = pd.concat(samples).reset_index(drop=True)
print(f"Val small: {len(val_small):,} rows")
print(val_small["Lane"].value_counts().to_string())
val_small.to_csv("Data/val_small.csv", index=False)
print("saved!")

Val small: 1,000 rows
Lane
SK1    250
1      250
2      250
3      250
saved!


In [3]:
# save as debug_nan.py and run it

import torch
import torch.nn as nn
import numpy as np
from dataset import get_train_loader
from model import get_model

print("Loading one batch...")
loader = get_train_loader(batch_size=4, num_workers=0)
images, gps, labels = next(iter(loader))

print(f"Images shape: {images.shape}")
print(f"Images min: {images.min():.4f}")
print(f"Images max: {images.max():.4f}")
print(f"Images has NaN: {torch.isnan(images).any()}")
print(f"Images has Inf: {torch.isinf(images).any()}")
print(f"GPS shape: {gps.shape}")
print(f"GPS values: {gps[0]}")
print(f"GPS has NaN: {torch.isnan(gps).any()}")
print(f"Labels: {labels}")

print("\nBuilding model...")
model = get_model(pretrained=False)
model.train()

print("Forward pass...")
logits = model(images, gps)
print(f"Logits: {logits}")
print(f"Logits has NaN: {torch.isnan(logits).any()}")

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
print(f"Loss: {loss.item()}")
print(f"Loss has NaN: {torch.isnan(loss).any()}")

C:\Users\gd\New folder\project\geosolve_lanes\geosolve_lanes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading one batch...
[Dataset] Loading Data\train_tiny.csv...
[Dataset] 500 valid images loaded (dropped ignore)
[Dataset] Lane distibution:
    1:        125 (25.0)
    2:        125 (25.0)
    3:        125 (25.0)
  SK1:        125 (25.0)
[DataLoader] Train : 500 images, 125 batches per epoch
Images shape: torch.Size([4, 3, 224, 224])
Images min: -2.1179
Images max: 2.6400
Images has NaN: False
Images has Inf: False
GPS shape: torch.Size([4, 5])
GPS values: tensor([ 1.0000,  0.0000,  0.0000, -0.7181, -0.6959])
GPS has NaN: False
Labels: tensor([0, 0, 0, 3])

Building model...
[Model] Backbone: efficientnet_b0
[Model] backbone output features: 1280
[Model] GPS processor: 5--> 64 features
[Model] Classifier: 1344 -> 4 classes
[Model] Output classes: 4
Forward pass...
Logits: tensor([[ 0.0697, -0.0997,  0.0587,  0.0914],
        [ 0.0485, -0.1661, -0.0934,  0.1333],
        [ 0.0711, -0.0953,  0.0619,  0.0807],
        [ 0.0846, -0.1092,  0.0896,  0.0720]], grad_fn=<AddmmBackward0>)
Log

In [5]:
# save as check_grad_clip.py and run
import inspect
import train
source = inspect.getsource(train.train_one_epoch)
if "clip_grad_norm" in source:
    print("Gradient clipping IS in train_one_epoch ✓")
else:
    print("Gradient clipping NOT found ✗")

if "CrossEntropyLoss()" in source or "CrossEntropyLoss(w" in source:
    print("CrossEntropyLoss found ✓")

Gradient clipping IS in train_one_epoch ✓


In [6]:
# save as debug_training.py
import torch
import torch.nn as nn
from dataset import get_train_loader
from model import get_model, freeze_backbone

loader = get_train_loader(batch_size=4, num_workers=0)
model  = get_model(pretrained=False)
freeze_backbone(model)
model.train()

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

print("Running 5 batches manually...")
for batch_idx, (images, gps, labels) in enumerate(loader):
    optimiser.zero_grad()
    logits = model(images, gps)
    loss   = criterion(logits, labels)

    print(f"Batch {batch_idx+1} | "
          f"logits min={logits.min():.4f} max={logits.max():.4f} | "
          f"loss={loss.item():.4f} | "
          f"loss isnan={torch.isnan(loss).item()}")

    if torch.isnan(loss):
        print(f"  NaN detected!")
        print(f"  Labels: {labels}")
        print(f"  Logits: {logits}")
        break

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimiser.step()

    if batch_idx >= 4:
        break

print("Done!")

[Dataset] Loading Data\train_tiny.csv...
[Dataset] 500 valid images loaded (dropped ignore)
[Dataset] Lane distibution:
    1:        125 (25.0)
    2:        125 (25.0)
    3:        125 (25.0)
  SK1:        125 (25.0)
[DataLoader] Train : 500 images, 125 batches per epoch
[Model] Backbone: efficientnet_b0
[Model] backbone output features: 1280
[Model] GPS processor: 5--> 64 features
[Model] Classifier: 1344 -> 4 classes
[Model] Output classes: 4
[Model] Backbone FROZEN - only classifier trains
[Model] total parameters: 4,764,672
[Model] trainable parameters: 757,124
Running 5 batches manually...
Batch 1 | logits min=-0.1363 max=0.0942 | loss=1.4174 | loss isnan=False
Batch 2 | logits min=-0.0751 max=0.0803 | loss=1.3967 | loss isnan=False
Batch 3 | logits min=-0.0884 max=0.0717 | loss=1.3764 | loss isnan=False
Batch 4 | logits min=-0.1167 max=0.0998 | loss=1.4066 | loss isnan=False
Batch 5 | logits min=-0.1322 max=0.0769 | loss=1.4128 | loss isnan=False
Done!


In [1]:
import pandas as pd

df = pd.read_csv("Data/val.csv")
print(df["Lane"].value_counts())

# stratified sample matching REAL distribution
# Lane 1: 96%, Lane 2: 3.6%, SK1: 0.2%, Lane 3: 0.2%
samples = []
targets = {"1": 2400, "2": 90, "SK1": 5, "3": 5}

for lane, n in targets.items():
    subset = df[df["Lane"] == lane]
    available = min(len(subset), n)
    samples.append(subset.sample(available, random_state=42))

val_real = pd.concat(samples).reset_index(drop=True)
print(f"Real distribution val set: {len(val_real):,} rows")
print(val_real["Lane"].value_counts())
val_real.to_csv("Data/val_real.csv", index=False)
print("saved!")

Lane
1      372995
2       10377
SK1      1470
3         685
Name: count, dtype: int64
Real distribution val set: 2,500 rows
Lane
1      2400
2        90
SK1       5
3         5
Name: count, dtype: int64
saved!


In [2]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("Data/train.csv", low_memory=False)
print(f"Total rows: {len(df):,}")

# split into 10 equal chunks
chunk_size = len(df) // 10
output_dir = Path("Data/chunks")
output_dir.mkdir(exist_ok=True)

for i in range(10):
    start = i * chunk_size
    end   = start + chunk_size if i < 9 else len(df)
    chunk = df.iloc[start:end].copy()

    output_path = output_dir / f"train_chunk_{i+1:02d}.csv"
    chunk.to_csv(output_path, index=False)
    print(f"Chunk {i+1}: {len(chunk):,} rows → {output_path}")

print("Done!")

Total rows: 2,291,725
Chunk 1: 229,172 rows → Data\chunks\train_chunk_01.csv
Chunk 2: 229,172 rows → Data\chunks\train_chunk_02.csv
Chunk 3: 229,172 rows → Data\chunks\train_chunk_03.csv
Chunk 4: 229,172 rows → Data\chunks\train_chunk_04.csv
Chunk 5: 229,172 rows → Data\chunks\train_chunk_05.csv
Chunk 6: 229,172 rows → Data\chunks\train_chunk_06.csv
Chunk 7: 229,172 rows → Data\chunks\train_chunk_07.csv
Chunk 8: 229,172 rows → Data\chunks\train_chunk_08.csv
Chunk 9: 229,172 rows → Data\chunks\train_chunk_09.csv
Chunk 10: 229,177 rows → Data\chunks\train_chunk_10.csv
Done!


In [3]:
import pandas as pd

df = pd.read_csv("Data/train.csv", low_memory=False)

lane1 = df[df["Lane"] == "1"].sample(200000, random_state=42)
lane2 = df[df["Lane"] == "2"]
lane3 = df[df["Lane"] == "3"]
sk1   = df[df["Lane"] == "SK1"]

balanced = pd.concat([lane1, lane2, lane3, sk1]).sample(frac=1, random_state=42)
print(f"Total: {len(balanced):,}")
print(balanced["Lane"].value_counts())
balanced.to_csv("Data/train_balanced.csv", index=False)
print("saved!")

Total: 297,930
Lane
1      200000
2       82741
3       10709
SK1      4480
Name: count, dtype: int64
saved!


In [6]:
import pandas as pd

df = pd.read_csv("J:/- Macros/AI-LaneDetector/lane_analysis/coordinates_report_with_lane_combined_with_ignore_full.csv", low_memory=False)

lane1 = df[df["Lane"] == "1"].sample(20, random_state=42)
lane2 = df[df["Lane"] == "2"].sample(10, random_state=42)
#lane3 = df[df["Lane"] == "3"].sample(10)
#sk1   = df[df["Lane"] == "SK1"]

balanced = pd.concat([lane1, lane2]).sample(frac=1, random_state=42)
df_new = balanced.drop(columns=['Lane'])
df_new.to_csv(r"J:/- Macros/AI-LaneDetector/lane_analysis/coordinates_report_with_lane_combined_with_ignore_full_output_for_predicting.csv", index=False)
print("saved!")

saved!
